# Pycavi illumination correction via Kriging read from file

This notebook is used to read an illumination correction determined from notebooks 8b using kriging which was previously saved as a FITS file. It also needs the illum.csv file that was used for illumination correction. This notebook is more of a sanity check that permits to compare the results obtained in notebook 8b, with what the corrections are when using the kriging results save to a FITS file.



After the illumination correctionis read it is shown as a colour image with contours overlaid.
The image can be saved to a file in the Calibration directory.

  - Two plots are generated for QC purposes: a residuals versus x-coordinates for the uncorrected points
in red, and the corrected points in black; and a histogram of the residuals for both
sets of points. The mean, median and 1-sigma variation is reported in these plots.
The plots can be saved.

  - Another set of plots is presented with the residuals as a function of radius, azimuthal
angle measured from the center of the FoV, and versus x and y. 


In [ ]:
import os
import sys
import glob
import copy
import logging

import numpy as np
from numpy.polynomial import chebyshev
import pandas as pd

#import matplotlib.pyplot as plt
#import matplotlib.gridspec as gridspec
#from matplotlib import cm
#from mpl_toolkits.axes_grid1 import make_axes_locatable

from astropy.stats import sigma_clipped_stats, sigma_clip
from astropy.visualization import simple_norm
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS

from scipy.interpolate import griddata, Rbf
from scipy.stats import norm

from photutils.aperture import SkyCircularAperture, SkyCircularAnnulus, aperture_photometry


#   from pycavi_astromatic import SExtractor
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production

from pycavi.pycavi import pycavi_tools
from pycavi.pycavi.pycavi_zpcal import extract_calibration, plot_zpcal, get_files_dict, save_illumination_data, read_illumination_data
from pycavi.pycavi.pycavi_zpcal import plot_residuals, plot_illumination_correction, plot_corrected_residuals, plot_spatial_residuals
from pycavi.pycavi.pycavi_chebyshev import fit_chebyshev, chebyshev_2d_eval, chebyshev_polynomials, evaluate_2dpoly, append_hashline
from pycavi.pycavi.pycavi_kriging import kriging_interpolation, determine_kriging_errors, save_kriging_model, get_kriging_interpolation


In [ ]:
# set the directory folders
config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)

observatory = config['Provenance']['observatory'] # Needed for the atmospheric extinction curve
project_dir = config['Directories']['project_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['reduced_science_dir']  
log_dir = config['Directories']['project_logs_dir']

log_file_swarp = log_dir + "/pymeffpp_zpcal_check.log"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_file_swarp),
                              logging.StreamHandler()])
logging.getLogger().setLevel(logging.INFO)

# At this point it is believed that the data reduction process
# should be closely monitored, so we calibrate target/filter sets,
# that is each data partition, one by one.
#
filters = []
reject_observations = []
dir_name = config['Directories']['cal_dir_name']
zpcal_dict = get_files_dict(reduced_std_dir, reduced_science_dir, dir_name, filters, reject_observations)

observation_tags = list(set(zpcal_dict.keys()))
observation_tags.sort()    # Needed to ensure same index in every run
illum_files = []
for i in range(len(observation_tags)):
    logging.debug("index: {} observation_tag: {}".format(i, observation_tags[i]))
    for j in range(len(zpcal_dict[observation_tags[i]])):
        directory = os.path.dirname(zpcal_dict[observation_tags[i]][j])
        file_name_noext = os.path.splitext(os.path.basename(zpcal_dict[observation_tags[i]][j]))[0]
        illum_file = os.path.join(directory, file_name_noext)+ '.illum.csv'
        if os.path.isfile(illum_file):
            illum_files.append(illum_file)

krige_file = {}
for i in range(len(illum_files)):
    logging.info("index: {} -- illum: {}".format(i, illum_files[i]))
    # for each illum file display the kriging fits files
    root_name = illum_files[i][:-10]
    file_list = glob.glob(root_name+'*.kriging.fits')
    if file_list:
        krige_file[i] = []
        j = 0
        for afile in file_list:
            krige_file[i].append(os.path.basename(afile))
            logging.info(f"        j:{j} -- > {os.path.basename(afile)}")
            j+=1



In [ ]:
# Here we loop over the selected csv files output of notebook 07b_pycavi_zpcal_qc
# See list above to select a data partition to calibrate
# For example, if the list is:
#
# index: 0 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/Calibration_dir/CP_Pup_r_SDSS.coadd.csv
# index: 1 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Calibration_dir/M67_g_SDSS.coadd.csv
# index: 2 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Calibration_dir/M67_r_SDSS.coadd.csv
# index: 3 -- file: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/Calibration_dir/M67_NB_659.coadd.csv
# index: 4 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Calibration_dir/SA101_g_SDSS.coadd.csv
# index: 5 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/Calibration_dir/SA101_r_SDSS.coadd.csv
# index: 6 -- file: /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/Calibration_dir/SA98_NB_659.coadd.csv
#
# then to calibrate M67_NB_659.coadd.csv set
#
# idx_selected = [3]
#

idx_selected = [1]    # Keep empty to check all files
kfile_idx = 1
for i in range(len(illum_files)):
    
    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue

    krige_fits_name = os.path.dirname(illum_files[i])+'/'+krige_file[i][kfile_idx]
    logging.info(f"krige_fits_name: {krige_fits_name}")
    
    # For checking we read it again and continue with plotting
    headers, data_df = read_illumination_data(illum_files[i])
    #display(data_df)
    print(data_df.columns)

    x_all = data_df['x_all'].to_numpy()
    y_all = data_df['y_all'].to_numpy()
    z_all = data_df['z_all'].to_numpy()
    z_all_err = data_df['z_all_err'].to_numpy()
    mag_std = data_df['mag_std'].to_numpy()

    dimx = int(headers['dimx'])
    dimy = int(headers['dimy'])
    origfile = headers['origfile']
    image_dir = headers['image_dir']
    image_name = headers['image_name']
    image_path = headers['image_path']
    band = headers['band']
    logging.info(f"image_name: {image_name}")
    
    brightest_mag = 12.0
    faintest_mag = 16.0
    dm_min = -0.25
    dm_max = 0.25

    z_all_clipped = sigma_clip(z_all, sigma=3.0, maxiters=5)
    mask = ~np.isnan(z_all) & ~z_all_clipped.mask & np.isfinite(z_all) & \
        (brightest_mag < mag_std) & (mag_std < faintest_mag) & \
        (dm_min < z_all) & (z_all < dm_max)
    x_subset = x_all[mask]
    y_subset = y_all[mask]
    z_subset = z_all[mask]
    z_subset_err = z_all_err[mask]

    z_mean_u, z_median_u, z_std_u = sigma_clipped_stats(z_subset, sigma=3.0, maxiters=5)
    logging.info("mean: {}  --  median: {}  --  stdev: {}".format(round(z_mean_u, 4), round(z_median_u, 4), round(z_std_u, 4)))

    n_before_filter = len(x_all)
    n_after_filter = len(x_subset)
    logging.info("Filtering: number of data points {} ---> {} ({} {})".format(n_before_filter, n_after_filter, dimx, dimy))
        

    
logging.info("-------Done-------")

In [ ]:
############################ 03 PLOT RESIDUALS AND PDF FOR UNCORRECTED DATA ####################
#
plot_residuals(x_subset, z_subset, ylim=(-0.25, 0.25))

In [ ]:
read_kriging_grid = True
if read_kriging_grid:
    
    # we also check the model by displaying it. The following cells need
    # the variables: x_subset, y_subset, z_subset, x_check, y_check, z_corr, grid_x, grid_y, grid_z, dimx, dimy, 
    # fit_plot, fit_image_title, res_plot_str, determine_errors, err_type should be None
    corr, grid_x, grid_y,  grid_z, zheader = get_kriging_interpolation(krige_fits_name, x_subset, y_subset)
    print(z_subset)
    dimx = zheader['ORIG_W']
    dimy = zheader['ORIG_H']
    x_check = x_subset
    y_check = y_subset
    z_corr = z_subset - corr
    fit_image_title = zheader['ORIGFILE']
    # Names for plots. Change if comparisson is desired.
    fit_plot = None
    save_res_plot_01 = None
    save_res_plot_02 = None

In [ ]:
########################################## 05 PLOT APPROXIMATING SURFACE ###########################
#
plot_illumination_correction(x_subset, y_subset, x_check, y_check, grid_x, grid_y, grid_z, dimx, dimy, fit_plot, fit_image_title)



In [ ]:
######################### PLOT RESIDUALS AND RMS ESTIMATE USING INDEPENDENT DATA POINTS ##################
#
#image_path = zpcal.image_path[:-5]
#image_name = image_name.split("/")[-1]

# In cases where edge oscilations are large  we might need to clip
# the max and minimum values.
min_corr = -0.20
max_corr = 0.25
#z_corr = np.clip(z_corr, min_corr, max_corr)
save_res_plot_01 = None
res_plot_str = image_name
determine_errors = True
err_type = "none"
plot_corrected_residuals(
    x_subset=x_subset, z_subset=z_subset, x_check=x_check, z_corr=z_corr,
    res_plot_str=res_plot_str, determine_errors=determine_errors,
    err_type=err_type, ylim=(-0.25, 0.25), save_plot=None
)



In [ ]:
############################ EXTENSIVE SPATIAL PLOTS OF RESIDUALS ###########################
#    
save_res_plot_01 = None
res_plot_str = image_name
determine_errors = True
err_type = "none"
plot_spatial_residuals(
    x_subset=x_subset, y_subset=y_subset, z_subset=z_subset,
    x_check=x_check, y_check=y_check, z_corr=z_corr,
    dimx=dimx, dimy=dimy, res_plot_str=res_plot_str,
    err_type=err_type, ylim=(-0.25, 0.25), save_plot=None
)


#_oOo_

### 